Changes:
- Previously, we were directly using the output of the crawler .
- The output has been transformed into a markdown file which is much better structured.
- The chunks will now consist of the md text.

In [10]:
import re
import numpy as np

def refined_chunking(text, chunk_size=1000, chunk_overlap=200):
    # Separatori ordonați după importanță (secțiune -> titlu -> paragraf -> linie -> frază)
    separators = ["\n---\n", "\n## ", "\n\n", "\n", ". ", " "]
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        # 1. Stabilim limita teoretică
        end = min(start + chunk_size, text_len)
        
        # 2. Căutăm cel mai bun separator în fereastra curentă
        if end < text_len:
            chosen_end = end
            for sep in separators:
                # Căutăm separatorul de la dreapta la stânga pentru a maximiza dimensiunea chunk-ului
                pos = text.rfind(sep, start, end)
                if pos != -1 and pos > start:
                    chosen_end = pos + len(sep)
                    break
            end = chosen_end

        # 3. Extragem chunk-ul
        chunk = text[start:end].strip()
        if len(chunk) > 20: 
            chunks.append(chunk)

        # Calculăm unde ar trebui să înceapă următorul chunk
        new_start = end - chunk_overlap
        
        # Dacă noul start nu avansează față de cel vechi, forțăm trecerea la 'end'
        # Asta se întâmplă când lungimea textului găsit e mai mică decât overlap-ul
        if new_start <= start:
            start = end
        else:
            start = new_start
            
    return chunks

# Citire și aplicare
with open('Data/MD file/admission_kb.md', 'r', encoding='utf-8') as f:
    text = f.read()

# Clean text
text = re.sub(r' +', ' ', text) 

chunks = refined_chunking(text)
print(f"Generat {len(chunks)} chunks optimizați.")

Generat 39 chunks optimizați.


In [11]:
# EMBEDDING (neschimbat)

from sentence_transformers import SentenceTransformer
import faiss  # baza de date specializata in viteza. Daca ai 1 milion de fragmente de text, FAISS (Facebook AI Similarity Search) gaseste instant care dintre ele seamana cu intrebarea ta
import numpy as np  # FAISS lucreaza doar cu structuri de date de tip numpy

#if model not in globals():   # daca nu am definit pana acum model
model = SentenceTransformer("intfloat/multilingual-e5-small")

# Adaugă prefixul obligatoriu pentru modelul E5 la indexare
chunks_with_prefix = [f"passage: {c}" for c in chunks]
embeddings = model.encode(chunks)

# Creeaza un index (memorie rapida) care masoara distanta "L2" (distanta euclidiana, ca o linie dreapta pe harta) intre vectori
index = faiss.IndexFlatL2(len(embeddings[0]))  # len(embeddings[0]) -> ii spune indexului cat de lung este un singur vector (pentru acest model in specific lungimea fiind de 384 de numere)
index.add(np.array(embeddings))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12105.56it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
# SAVE
np.save("Data/chunks.npy", chunks)
faiss.write_index(index, "Data/index.faiss")